# Risø Conference Proceedings 2026 - grazing-incidence case
This notebook contains the accompanying processing code for the grazing-indidence case to support the Risø Conference Proceedings 2026 paper:

Ball, J. A. D., Andreasen, J. W., Angelis, S. D., Wright, J. P., & Detlefs, C. (2026, July 9). Multi-Beam 3DXRD. IOP Conference Series: Materials Science and Engineering. 46th Risø International Symposium on Materials Science: Characterization of evolving microstructures in metals, DTU Risø Campus, Roskilde, Denmark. Accepted for publication.

In [ ]:
import os
# don't hog GPU memory (if you have one)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "0"

import jax
jax.config.update("jax_enable_x64", True)  # required for accuracy, especially strains
import jax.numpy as jnp
import optax

import time
from functools import partial

import scipy
from scipy.spatial.transform import Rotation
from matplotlib import pyplot as plt

import ImageD11.grain
import ImageD11.unitcell
import ImageD11.indexing
from xfab.symmetry import ROTATIONS, Umis

from ImageD11.nbGui.nb_utils import plot_grain_positions, plot_all_ipfs

import anri.crystal, anri.diffract, anri.geom, anri.fwd

start = time.time()

In [ ]:
### BEAM

# Energies in eV
beam_ev = 12_000

def eV_to_wavelength_A(energy_eV):
    # L = hc/E
    # Must convert eV to Joules first
    wavelength_A = (scipy.constants.h * scipy.constants.c)/(scipy.constants.e*energy_eV)*1e10
    return wavelength_A

wavelength = eV_to_wavelength_A(beam_ev)

wavelength

## Crystallography
We'll use cubic Si here

In [ ]:
struc = anri.crystal.Structure.from_cif("../../../tests/data/cif/Si.cif")
struc.lattice_parameters

We generate some HKLs.

In [ ]:
struc.make_hkls(dsmax=1.0, wavelength=wavelength)
struc.rings_table

## Phantom sample

Let's define our phantom sample.  
We define everything in the sample coordinate system (on top of the goniometer).  
We'll randomly generate some grains within a box.

In [ ]:
rng = 54  # chosen by fair dice roll, guaranteed to be random
key = jax.random.key(rng)

n_grains = 100

### Positions
sample_width = 1000.0
sample_height = 10.0
translations_sample_xy = jax.random.uniform(key, shape=(n_grains,2,), minval=-sample_width/2, maxval=sample_width/2)
translations_sample_z  = jax.random.uniform(key, shape=(n_grains,), minval=-sample_height/2, maxval=sample_height/2)
translations_sample = jnp.column_stack((translations_sample_xy, translations_sample_z))

### Orientations
U_matrices = Rotation.random(n_grains, rng=rng).as_matrix()
UB_matrices = U_matrices @ struc.B
UBI_matrices = jnp.linalg.inv(UB_matrices)

### Volumes - just for visualisation purposes!
radii_sigma = 0.2
radii_mean = 50.0
radii = jax.random.lognormal(key, shape=(n_grains,), sigma=radii_sigma) * radii_mean
volumes = (4./3)*jnp.pi*(radii**3)

We can use ImageD11 to plot this phantom.

In [ ]:
ref_unitcell = ImageD11.unitcell.unitcell(struc.lattice_parameters, symmetry=struc.sgno)
grains = [ImageD11.grain.grain(UBI_matrices[i], translation=translations_sample[i]) for i in range(n_grains)]

for i, g in enumerate(grains):
    g.ref_unitcell = ref_unitcell
    g.intensity_info = f"mean = {volumes[i]}"

plot_grain_positions(grains, 'z', size_scaling=0.1)
plot_all_ipfs(grains)

## Forward projection
Now we can generate our scattering vectors and translate them into the lab frame, then into the detector.  
This will yield centroids, which are arrays of `[slow, fast, omega]` which represent the centre-of-mass positions of the peaks on the detector surface.  
We must define our goniometer and detector positions:

In [ ]:
### Goniometer

wedge = -0.5  # anri now right-handed!
chi = 0.0
y0 = 0.0

### Detector
y_center = 1024.0
z_center = -1024.0
y_size = 50.0
z_size = 50.0
tilt_x = 0.0
tilt_y = jnp.radians(-60.0)
tilt_z = 0.0
distance = 180e3
o11 = 1
o12 = 0
o21 = 0
o22 = 1

# not official parameters but useful for plotting later
det_size_s = 2048
det_size_f = 2048

# Get change-of-basis parameters to go from lab to detector space:
det_trans, beam_cen_shift, x_distance_shift = anri.geom.detector_transforms(
    y_center,
    y_size,
    tilt_y,
    z_center,
    z_size,
    tilt_z,
    tilt_x,
    distance,
    o11,
    o12,
    o21,
    o22
)

# Get detector unit vectors (slow, fast directions) in lab frame:
sc_lab, fc_lab, norm_lab = anri.geom.detector_basis_vectors_lab(det_trans, beam_cen_shift, x_distance_shift)

In [ ]:
# we get plus/minus Friedel pairs
# we also get boolean masks for validity - sometimes no solution exists for the Ewald condition, so we never see the peak.

k_in_direct = jnp.array([1., 0, 0])  # direct beam wavevector
k_in_refl = anri.geom.rot_y(2*wedge) @ k_in_direct  # reflected beam wavevector


centroid_dir_p, valid_dir_p = anri.fwd.get_centroid_box_all(UBI_matrices, translations_sample,
                                              struc.ringhkls_arr,
                                              1.0, wavelength, k_in_direct, 0, 0,
                                              wedge, chi,
                                              sc_lab, fc_lab, norm_lab)
centroid_dir_m, valid_dir_m = anri.fwd.get_centroid_box_all(UBI_matrices, translations_sample,
                                              struc.ringhkls_arr,
                                              -1.0, wavelength, k_in_direct, 0, 0,
                                              wedge, chi,
                                              sc_lab, fc_lab, norm_lab)

centroid_refl_p, valid_refl_p = anri.fwd.get_centroid_box_all(UBI_matrices, translations_sample,
                                              struc.ringhkls_arr,
                                              1.0, wavelength, k_in_refl, 0, 0,
                                              wedge, chi,
                                              sc_lab, fc_lab, norm_lab)
centroid_refl_m, valid_refl_m = anri.fwd.get_centroid_box_all(UBI_matrices, translations_sample,
                                              struc.ringhkls_arr,
                                              -1.0, wavelength, k_in_refl, 0, 0,
                                              wedge, chi,
                                              sc_lab, fc_lab, norm_lab)

# join Friedel pairs into single datasets:
centroid_dir = jnp.concatenate([centroid_dir_p[valid_dir_p], centroid_dir_m[valid_dir_m]])
centroid_refl = jnp.concatenate([centroid_refl_p[valid_refl_p], centroid_refl_m[valid_refl_m]])

# flatten into (N, 3)
centroid_dir = centroid_dir.reshape(-1, 3)
centroid_refl = centroid_refl.reshape(-1, 3)

# mask to detector pixel range
m_dir = (centroid_dir[:, 0] > 0) & (centroid_dir[:, 0] < det_size_f) & (centroid_dir[:, 1] > 0) & (centroid_dir[:, 1] < det_size_s)
m_refl = (centroid_refl[:, 0] > 0) & (centroid_refl[:, 0] < det_size_f) & (centroid_refl[:, 1] > 0) & (centroid_refl[:, 1] < det_size_s)
centroid_dir = centroid_dir[m_dir]
centroid_refl = centroid_refl[m_refl]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,20), constrained_layout=True)
axs[0].scatter(centroid_dir[:, 1], centroid_dir[:, 0], label=r'Direct peaks', s=2)
axs[0].scatter(centroid_refl[:, 1], centroid_refl[:, 0], label=r'Reflected peaks', s=2)
axs[0].set_aspect(1)
axs[0].set(xlabel='Detector fast', ylabel='Detector slow', title='Whole detector view', xlim=(0, det_size_f), ylim=(0, det_size_s))
axs[0].legend(loc='upper right')

axs[1].scatter(centroid_dir[:, 1], centroid_dir[:, 0], label=r'Direct peaks')
axs[1].scatter(centroid_refl[:, 1], centroid_refl[:, 0], label=r'Reflected peaks')
axs[1].set_aspect(1)
axs[1].set(xlabel='Detector fast', ylabel='Detector slow', xlim=(300, 450), ylim=(850, 1000), title='Detail view')
axs[1].legend(loc='upper right')
plt.show()

## Scattering vector identification
With the peaks forward-projected onto the detector, we now 'forget' which wavelength each peak came from.  
We compute scattering vectors in the sample frame for *all* peaks, twice, assuming each wavelength.  
We also forget the origin of diffraction of each centroid.

In [ ]:
centroid = jnp.concatenate([centroid_dir, centroid_refl])

## Simulating experimental error
We now make a sensible effort to "spoil" the measured centroids to account for experimental errors.  
This can be done more accurately (and will be in the future when we simulate intensity profiles on the detector) but to first order this should be a reasonable approach.

In [ ]:
def spoil_centroids(centroids, width_px, width_omega):
    npks = centroids.shape[0]
    px_error_vector = jax.random.uniform(key, shape=(npks,2), minval=-width_px/2, maxval=width_px/2)
    omega_error_vector = jax.random.uniform(key, shape=(npks,), minval=-width_omega/2, maxval=width_omega/2)
    error_vector = jnp.column_stack((px_error_vector, omega_error_vector))
    centroids = centroids + error_vector
    return centroids

ostep = 0.1  # realistic omega step
centroid_with_error = spoil_centroids(centroid, 1.0, ostep)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,20), constrained_layout=True)
axs[0].scatter(centroid[:, 1], centroid[:, 0], label=r'Peaks without error')
axs[0].scatter(centroid_with_error[:, 1], centroid_with_error[:, 0], s=10, label=r'Peaks with error')
axs[0].set_aspect(1)
axs[0].set(xlabel='Detector fast', ylabel='Detector slow', title='Whole detector view')
axs[0].legend(loc='upper right')

axs[1].scatter(centroid[:, 1], centroid[:, 0], label=r'Peaks without error')
axs[1].scatter(centroid_with_error[:, 1], centroid_with_error[:, 0], s=10, label=r'Peaks with error')
axs[1].set_aspect(1)
axs[1].set(xlabel='Detector fast', ylabel='Detector slow', xlim=(300, 450), ylim=(850, 1000), title='Detail view')
axs[1].legend(loc='upper right')
plt.show()

In [ ]:
# All these primitive functions are written for single vectors
# We write the overall function for a single vector, then vmap it over many vectors for speed:

@jax.jit
def detector_to_q(slow, fast, omega, wavelength, k_in_lab_hat, origin_sample):
    # peak vector in lab frame
    peak_lab = anri.geom.det_to_lab(slow, fast, det_trans, beam_cen_shift, x_distance_shift)
    # rotate origin into sample frame
    origin_lab = anri.geom.sample_to_lab(origin_sample, omega, wedge, chi, 0.0, 0.0)
    
    # convert peak vector to k_out, subtracts off the origin
    k_out_lab_norm = anri.diffract.peak_lab_to_k_out(peak_lab, origin_lab, wavelength)
    # normalise k_in by wavelength
    k_in_lab_norm = anri.diffract.scale_norm_k(k_in_lab_hat, wavelength)
    # simply q = k_out - k_in
    q_lab = anri.diffract.k_to_q_lab(k_in_lab_norm, k_out_lab_norm)
    # rotate into sample frame
    q_sample = anri.geom.lab_to_sample(q_lab, omega, wedge, chi, 0.0, 0.0)
    return q_sample

# the vmap operation
detector_to_q_vec = jax.vmap(detector_to_q, in_axes=(0, 0, 0, 0, 0, None))

In [ ]:
# Compute scattering vectors for all peaks assuming each beam direction

kvecs_direct = jnp.broadcast_to(k_in_direct, centroid_with_error.shape)
kvecs_refl = jnp.broadcast_to(k_in_refl, centroid_with_error.shape)
wavelengths = jnp.full(centroid_with_error.shape[0], wavelength)

q_sample_direct = detector_to_q_vec(centroid_with_error[:, 0], centroid_with_error[:, 1], centroid_with_error[:, 2], wavelengths, kvecs_direct, jnp.array([0., 0., 0.]))
q_sample_refl   = detector_to_q_vec(centroid_with_error[:, 0], centroid_with_error[:, 1], centroid_with_error[:, 2], wavelengths, kvecs_refl, jnp.array([0., 0., 0.]))

## *k*-d tree search
We now construct a *k*-d tree in sample space.  
The idea is that half of the observed scattering vectors will be correctly computed with the direct beam k_in vector, and the other half will be correctly computed with the reflected beam k_in vector.  
As these duplicated scattering vectors come from a single set of reciprocal lattice points, the two datasets should 'overlap' if and only if the incoming wavevector was correctly chosen for a given scattering vector.
We look for 'overlapping' (i.e. duplicate) scattering vectors in sample space using a *k*-d tree.

In [ ]:
kd_refl = scipy.spatial.cKDTree(q_sample_refl)

# Find the pairs
distances, indices = kd_refl.query(q_sample_direct, k=1, distance_upper_bound=0.01)
valid_mask = jnp.isfinite(distances)
valid_distances = distances[valid_mask]

fig, ax = plt.subplots()
ax.hist(valid_distances, bins=100)
ax.set(xlabel='Distance in g-vector space', ylabel='Count', title='Histogram of g-vector neighbour distances')
plt.show()

We can perhaps discern that a sensible cutoff would be `0.0015` to get the first spike

In [ ]:
# Find the pairs
distances, indices = kd_refl.query(q_sample_direct, k=1, distance_upper_bound=0.0015)
valid_mask = jnp.isfinite(distances)
valid_distances = distances[valid_mask]

fig, ax = plt.subplots()
ax.hist(valid_distances, bins=50)
ax.set(xlabel='Distance in g-vector space', ylabel='Count', title='Histogram of g-vector neighbour distances')
plt.show()

Now we need to average or combine our observations of the g-vectors.

We have three sources of g-vectors we can possibly index:

- G-vectors only from $K_{\alpha_{1}}$
- G-vectors only from $K_{\alpha_{2}}$
- Averaged/combined observations of both

In [ ]:
mask_dir = valid_mask
mask_refl = indices[mask_dir]

q_sample_direct_paired = q_sample_direct[mask_dir]
q_sample_refl_paired = q_sample_refl[mask_refl]

# concatenate
q_sample_cat = jnp.concatenate((q_sample_direct_paired, q_sample_refl_paired))

We can confirm that the deduplication succeeded by masking the centroids array in the same way:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,20), constrained_layout=True)
axs[0].scatter(centroid_with_error[:, 1], centroid_with_error[:, 0], s=50, label=r'Direct and reflected beam peaks')
axs[0].scatter(centroid_with_error[:, 1][mask_dir], centroid_with_error[:, 0][mask_dir], s=10, label=r'Deduplicated peaks')
axs[0].set_aspect(1)
axs[0].set(xlabel='Detector fast', ylabel='Detector slow', title='Whole detector view', xlim=(0, det_size_f), ylim=(0, det_size_s))
axs[0].legend(loc='upper right')

axs[1].scatter(centroid_with_error[:, 1], centroid_with_error[:, 0], s=50, label=r'Direct and reflected beam peaks')
axs[1].scatter(centroid_with_error[:, 1][mask_dir], centroid_with_error[:, 0][mask_dir], s=10, label=r'Deduplicated peaks')
axs[1].set_aspect(1)
axs[1].set(xlabel='Detector fast', ylabel='Detector slow', xlim=(300, 450), ylim=(850, 1000), title='Detail view')
axs[1].legend(loc='upper right')
plt.show()

## Indexing the recovered scattering vectors
### Just direct beam

In [ ]:
# make an indexer
idx_dir = ImageD11.indexing.indexer(unitcell=ref_unitcell, gv=q_sample_direct_paired, minpks=15)
idx_dir.ds_tol = 0.005
idx_dir.assigntorings()
idx_dir.hkl_tol = 0.025
idx_dir.cosine_tol = 0.002
idx_dir.score_all_pairs()
idx_dir.saveubis('grains_found_direct.ubi')

### Just reflected beam

In [ ]:
# make an indexer
idx_refl = ImageD11.indexing.indexer(unitcell=ref_unitcell, gv=q_sample_refl_paired, minpks=15)
idx_refl.ds_tol = 0.005
idx_refl.assigntorings()
idx_refl.hkl_tol = 0.025
idx_refl.cosine_tol = 0.002
idx_refl.score_all_pairs()
idx_refl.saveubis('grains_found_refl.ubi')

### Combined scattering vectors

In [ ]:
# make an indexer
idx_both = ImageD11.indexing.indexer(unitcell=ref_unitcell, gv=q_sample_cat, minpks=30)
idx_both.ds_tol = 0.005
idx_both.assigntorings()
idx_both.hkl_tol = 0.025
idx_both.cosine_tol = 0.002
idx_both.score_all_pairs()
idx_both.saveubis('grains_found_both.ubi')

## Grain position and UBI refinement
We can now try to refine the positions and UBIs of the grains we've found.  
We do this in three ways:  
- Indexed with direct beam g-vectors, refined with direct beam centroids
- Indexed with reflected beam g-vectors, refined with reflected beam centroids
- Indexed with combined g-vectors, refined with both direct and reflected beam centroids (alternating)

We implement a gradient-aware minimiser using ADAM from the Optax library. The loss function itself is identical to `makemap.py` in ImageD11.

### Peak to grain assignment
We have to establish static peak to grain assignments before we can refine, so grains do not compete for peaks. We do this just like ImageD11 - each peak is assigned to the grain that best indexes it (yields $hkl$ values closest to integer).

In [ ]:
@jax.jit
def assign(ubi, q_sample):
    hklf = (ubi @ q_sample.T).T
    hkli = jnp.rint(hklf)
    hkle = jnp.linalg.norm(hklf - hkli)
    return hkle, hkli

assign_peaks = jax.vmap(assign, in_axes=[None, 0])
assign_grains = jax.vmap(assign_peaks, in_axes=[0, None])

# compute hkl errors
hkle_dir,  hkli_dir  = assign_grains(jnp.array(idx_dir.ubis),  q_sample_direct[mask_dir])
hkle_refl, hkli_refl = assign_grains(jnp.array(idx_refl.ubis), q_sample_refl[mask_refl])
hkle_both, hkli_both = assign_grains(jnp.array(idx_both.ubis), q_sample_cat)

# get grain assignments (minimised error per peak)
ga_dir  = jnp.argmin(hkle_dir, axis=0)
ga_refl = jnp.argmin(hkle_refl, axis=0)
ga_both = jnp.argmin(hkle_both, axis=0)

# get hkli per peak
hkli_dir  = jnp.squeeze(jnp.take_along_axis(hkli_dir,  ga_dir[None, :, None], axis=0))
hkli_refl = jnp.squeeze(jnp.take_along_axis(hkli_refl, ga_refl[None, :, None], axis=0))
hkli_both = jnp.squeeze(jnp.take_along_axis(hkli_both, ga_both[None, :, None], axis=0))

# max grains per peak
M = max(jnp.unique(ga_dir, return_counts=True)[1].max(), jnp.unique(ga_refl, return_counts=True)[1].max(), jnp.unique(ga_both, return_counts=True)[1].max())
M

In [ ]:
@jax.jit
def solve_ub_analytical(hkl_int, q_sample, mask):
    """q = UB @ h  =>  UB = (Q^T H)(H^T H)^-1, masked rows contributing zero."""
    m = mask.astype(q_sample.dtype)[:, None]

    H  = hkl_int  * m
    Qs = q_sample * m

    HHT  = H.T @ H          # (3, 3), symmetric
    QsHT = Qs.T @ H

    # degenerate grains would otherwise NaN one row of the vmapped batch
    HHT = HHT + (1e-12 * jnp.trace(HHT) + 1e-15) * jnp.eye(3, dtype=HHT.dtype)

    return jnp.linalg.solve(HHT, QsHT.T).T


@jax.jit
def get_grain_loss_function(ubi, origin_sample, cen_obs, hkl_int, wavelength, kvecs, mask):
    # NOTE: `ubi` is deliberately unused — UB is profiled out analytically below.
    q_sample = detector_to_q_vec(cen_obs[:, 0], cen_obs[:, 1], cen_obs[:, 2],
                                 wavelength, kvecs, origin_sample)

    ub_fit  = solve_ub_analytical(hkl_int, q_sample, mask)
    ubi_fit = jnp.linalg.inv(ub_fit)

    d    = jnp.linalg.solve(ub_fit, q_sample.T).T - hkl_int
    hkle = jnp.sqrt(jnp.sum(d * d, axis=1) + 1e-24)      # NaN-safe at zero residual

    mf = mask.astype(hkle.dtype)
    return jnp.sum(hkle * mf) / (jnp.sum(mf) + 1e-10) + 1e-10, ubi_fit


@partial(jax.jit, static_argnums=(7,))
def refine_grain(initial_ubi, initial_origin, cen_obs, hkl_int, wavelength, kvecs, mask,
                 num_steps=100, learning_rate=1e-2, scaling_factor=1000.0):

    x0 = initial_origin / scaling_factor 

    optimizer  = optax.adam(learning_rate=learning_rate)
    opt_state0 = optimizer.init(x0)

    def objective(scaled_origin):
        loss, ubi = get_grain_loss_function(
            initial_ubi, scaled_origin * scaling_factor,
            cen_obs, hkl_int, wavelength, kvecs, mask)
        return loss, ubi

    grad_fn = jax.value_and_grad(objective, has_aux=True)

    def scan_body(carry, _):
        x, opt_state, best_x, best_loss = carry

        (loss, _), grads = grad_fn(x)

        better    = loss < best_loss
        best_x    = jnp.where(better, x, best_x)
        best_loss = jnp.where(better, loss, best_loss)

        updates, opt_state = optimizer.update(grads, opt_state)
        x = optax.apply_updates(x, updates)

        return (x, opt_state, best_x, best_loss), loss

    init_carry = (x0, opt_state0, x0, jnp.asarray(jnp.inf, x0.dtype))
    (_, _, best_x, _), loss_history = jax.lax.scan(
        scan_body, init_carry, None, length=num_steps)

    final_origin = best_x * scaling_factor

    final_loss, final_ubi = get_grain_loss_function(
        initial_ubi, final_origin, cen_obs, hkl_int, wavelength, kvecs, mask)

    loss_history = jnp.concatenate([loss_history, final_loss[None]])

    return final_ubi, final_origin, loss_history

refine_vmap = jax.vmap(
    refine_grain, 
    in_axes=(0, 0, 0, 0, 0, 0, 0, None, None, None)
)

### Just direct beam

In [ ]:
%%time

all_masks_dir = jnp.array([ga_dir == gid for gid in range(len(idx_dir.ubis))])
all_origins_dir = jnp.zeros((len(idx_dir.ubis), 3))

def get_compressed_grain_data(grain_id, assignments, cen, hkl, wave, kvec):
    # Create a boolean mask for this specific grain
    grain_mask = (assignments == grain_id)
    
    # Use top_k to find the indices of the True values.
    # This identifies exactly which rows in the filtered arrays belong to this grain.
    # Since we want static shapes, we always take M indices.
    _, indices = jax.lax.top_k(grain_mask.astype(kvec.dtype), M)
    
    # Gather the data for this grain
    # If the grain has < M peaks, top_k will pad with the last found index, 
    # but the 'mask' will correctly be False for those duplicates.
    return {
        "cen": cen[indices],
        "hkl": hkl[indices],
        "wave": wave[indices],
        "kvec": kvec[indices],
        "mask": grain_mask[indices]
    }

# Vectorize the packing over all grain IDs
v_pack = jax.vmap(
    get_compressed_grain_data, 
    in_axes=(0, None, None, None, None, None)
)

# Pack the data into grain-specific buffers (num_grains, M, ...)
grain_ids_dir = jnp.arange(len(idx_dir.ubis))
compressed_dir   = v_pack(grain_ids_dir,   ga_dir,   centroid_with_error[mask_dir], hkli_dir,   wavelengths, kvecs_direct)
# …likewise compressed_refl, compressed_both

# Run the refinement
# All inputs (except scalars) are now (num_grains, ...) so in_axes are all 0
ubis_fit_dir, origins_sample_fit_dir, all_losses_dir = refine_vmap(
    jnp.array(idx_dir.ubis),
    all_origins_dir,
    compressed_dir["cen"],
    compressed_dir["hkl"],
    compressed_dir["wave"],
    compressed_dir["kvec"],
    compressed_dir["mask"],
    100,
    1e-2,
    1000.0
)

ubis_fit_dir.block_until_ready()

In [ ]:
fig, axs = plt.subplots(2,1,sharex=True,sharey=True)
axs[0].plot(all_losses_dir.T)
axs[1].plot(all_losses_dir.mean(axis=0), label='dir')
axs[0].set(yscale='log')
axs[1].set(yscale='log')
axs[0].set(title='Loss over all grains')
axs[1].set(xlabel='Epoch', ylabel='Loss',title='Mean loss')
plt.show()

### Just reflected beam

In [ ]:
%%time

all_masks_refl = jnp.array([ga_refl == gid for gid in range(len(idx_refl.ubis))])
all_origins_refl = jnp.zeros((len(idx_refl.ubis), 3))

# Pack the data into grain-specific buffers (num_grains, M, ...)
grain_ids_refl = jnp.arange(len(idx_refl.ubis))
compressed_refl = v_pack(grain_ids_refl, ga_refl, centroid_with_error[mask_refl], hkli_refl, wavelengths, kvecs_refl)

# 2. Run the refinement
# All inputs (except scalars) are now (num_grains, ...) so in_axes are all 0
ubis_fit_refl, origins_sample_fit_refl, all_losses_refl = refine_vmap(
    jnp.array(idx_refl.ubis),
    all_origins_refl,
    compressed_refl["cen"],
    compressed_refl["hkl"],
    compressed_refl["wave"],
    compressed_refl["kvec"],
    compressed_refl["mask"],
    100,
    1e-2,
    1000.0
)

ubis_fit_refl.block_until_ready()

### Both direct and reflected beams

In [ ]:
%%time

all_masks_both = jnp.array([ga_both == gid for gid in range(len(idx_both.ubis))])
all_origins_both = jnp.zeros((len(idx_both.ubis), 3))
centroids_both = jnp.concatenate([centroid_with_error[mask_dir], centroid_with_error[mask_refl]])
wavelengths_both = jnp.concatenate([wavelengths[mask_dir], wavelengths[mask_refl]])
kvecs_both = jnp.concatenate([kvecs_direct[mask_dir], kvecs_refl[mask_refl]])

# Pack the data into grain-specific buffers (num_grains, M, ...)
grain_ids_both = jnp.arange(len(idx_both.ubis))
compressed_both = v_pack(grain_ids_both, ga_both, centroids_both, hkli_both, wavelengths_both, kvecs_both)

# Run the refinement
ubis_fit_both, origins_sample_fit_both, all_losses_both = refine_vmap(
    jnp.array(idx_both.ubis),
    all_origins_both,
    compressed_both["cen"],
    compressed_both["hkl"],
    compressed_both["wave"],
    compressed_both["kvec"],
    compressed_both["mask"],
    100,
    1e-2,
    1000.0
)

ubis_fit_both.block_until_ready()

In [ ]:
fig, ax = plt.subplots()
ax.plot(all_losses_dir.mean(axis=0), label='Direct beam')
ax.plot(all_losses_refl.mean(axis=0), label='Reflected beam')
ax.plot(all_losses_both.mean(axis=0), label='Both beams')
ax.set(yscale='log')
ax.legend()
ax.set(xlabel='Epoch', ylabel='loss', title='Loss over all grains')
plt.show()

## Match results to ground truth
As the indexer returns UBIs in a different order, we need to match the refinement results to the ground-truth grains.  
First, we make grain lists from the refined results.  
Then, we map each of the grain lists (including the ground truth grains) into the fundamental zone by maximising traces.  
Then we look for matches with a 6D translation-orientation feature vector $k$-d tree. This is highly biased towards orientations because they should more more reliable than translations.

In [ ]:
SYM = jnp.array(ROTATIONS[7])

def cast_to_fundamental_zone(U, symmetry_ops):
    best_U = U
    max_trace = jnp.trace(U)
    
    for S in symmetry_ops:

        U_equiv = jnp.matmul(U, S)
        
        current_trace = jnp.trace(U_equiv)
        
        if current_trace > max_trace:
            max_trace = current_trace
            best_U = U_equiv
            
    return best_U

def move_grains_to_fz(gl, sym_ops):
    newgl = []
    for g in gl:
        best_U = cast_to_fundamental_zone(g.U, sym_ops)
        new_UB = best_U @ g.B
        new_UBI = jnp.linalg.inv(new_UB)
        newg = ImageD11.grain.grain(new_UBI, translation=g.translation)
        newg.ref_unitcell = ref_unitcell
        newgl.append(newg)

    return newgl

def match_grains(grains1, grains2, W=50, dist_tol=100):
    """Returns (distance, index into grains1, index into grains2), all same length."""
    def desc(gl):
        return jnp.column_stack([
            jnp.array([jnp.asarray(g.translation) for g in gl]),
            W * jnp.array([Rotation.from_matrix(jnp.asarray(g.U)).as_rotvec() for g in gl])])
    d, m = scipy.spatial.cKDTree(desc(grains2)).query(desc(grains1), k=1,
                                                      distance_upper_bound=dist_tol)
    valid = jnp.isfinite(d)
    return d[valid], jnp.nonzero(valid)[0], m[valid]

@jax.jit
def gt_ubi_aligned(U_gt, U_fit, B):
    """GT UBI in the same symmetry setting as the fitted grain."""
    M = U_fit.T @ U_gt
    S = SYM[jnp.argmax(jnp.trace(M @ SYM, axis1=1, axis2=2))]
    return jnp.linalg.inv((U_gt @ S) @ B)

@jax.jit
def loss_at_fixed_ubi(ubi, origin, cen, hkl, wave, kvec, mask):
    """Mean |dhkl| for a GIVEN ubi — no analytical UB refit."""
    q = detector_to_q_vec(cen[:, 0], cen[:, 1], cen[:, 2], wave, kvec, origin)
    d = (ubi @ q.T).T - hkl
    e = jnp.sqrt(jnp.sum(d * d, axis=1) + 1e-24)
    mf = mask.astype(e.dtype)
    return jnp.sum(e * mf) / (jnp.sum(mf) + 1e-10) + 1e-10

gt_loss_vmap = jax.vmap(loss_at_fixed_ubi, in_axes=(0, 0, 0, 0, 0, 0, 0))

# ---------- refined grain lists ----------

grains_dir  = [ImageD11.grain.grain(ubis_fit_dir[g],  translation=origins_sample_fit_dir[g])  for g in range(len(idx_dir.ubis))]
grains_refl = [ImageD11.grain.grain(ubis_fit_refl[g], translation=origins_sample_fit_refl[g]) for g in range(len(idx_refl.ubis))]
grains_both = [ImageD11.grain.grain(ubis_fit_both[g], translation=origins_sample_fit_both[g]) for g in range(len(idx_both.ubis))]

oriens_dir  = jnp.stack([g.U for g in grains_dir])
oriens_refl = jnp.stack([g.U for g in grains_refl])
oriens_both = jnp.stack([g.U for g in grains_both])

# ---------- match ----------

grains_fz      = move_grains_to_fz(grains,      ROTATIONS[7])
grains_dir_fz  = move_grains_to_fz(grains_dir,  ROTATIONS[7])
grains_refl_fz = move_grains_to_fz(grains_refl, ROTATIONS[7])
grains_both_fz = move_grains_to_fz(grains_both, ROTATIONS[7])

dist_dir,  fit_dir,  gt_dir  = match_grains(grains_dir_fz,  grains_fz)
dist_refl, fit_refl, gt_refl = match_grains(grains_refl_fz, grains_fz)
dist_both, fit_both, gt_both = match_grains(grains_both_fz, grains_fz)

for tag, fit, gt, n in [('a1', fit_dir, gt_dir, len(grains_dir)),
                        ('a2', fit_refl, gt_refl, len(grains_refl)),
                        ('both', fit_both, gt_both, len(grains_both))]:
    print(f"{tag}: {len(fit)}/{n} matched, {len(gt) - len(jnp.unique(gt))} GT grains claimed twice")

# ---------- ground-truth losses ----------

def gt_losses(gt_idx, fit_idx, oriens, packed):
    ubi_gt = jax.vmap(gt_ubi_aligned, in_axes=(0, 0, None))(
        U_matrices[gt_idx], oriens[fit_idx], struc.B)
    return gt_loss_vmap(ubi_gt, translations_sample[gt_idx],
                        packed["cen"][fit_idx],  packed["hkl"][fit_idx],
                        packed["wave"][fit_idx], packed["kvec"][fit_idx],
                        packed["mask"][fit_idx])

gt_losses_dir  = gt_losses(gt_dir,  fit_dir,  oriens_dir,  compressed_dir)
gt_losses_refl = gt_losses(gt_refl, fit_refl, oriens_refl, compressed_refl)
gt_losses_both = gt_losses(gt_both, fit_both, oriens_both, compressed_both)

# ---------- metrics ----------

pos_diff_dir  = jnp.linalg.norm(translations_sample[gt_dir]  - origins_sample_fit_dir[fit_dir],   axis=1)
pos_diff_refl = jnp.linalg.norm(translations_sample[gt_refl] - origins_sample_fit_refl[fit_refl], axis=1)
pos_diff_both = jnp.linalg.norm(translations_sample[gt_both] - origins_sample_fit_both[fit_both], axis=1)

misorien_dir  = jnp.array([jnp.min(Umis(U_matrices[g], oriens_dir[f],  7)[:, 1]) for g, f in zip(gt_dir,  fit_dir)])
misorien_refl = jnp.array([jnp.min(Umis(U_matrices[g], oriens_refl[f], 7)[:, 1]) for g, f in zip(gt_refl, fit_refl)])
misorien_both = jnp.array([jnp.min(Umis(U_matrices[g], oriens_both[f], 7)[:, 1]) for g, f in zip(gt_both, fit_both)])

strains_dir  = jnp.array([grains_dir[f].eps_sample_matrix(struc.lattice_parameters)  for f in fit_dir])
strains_refl = jnp.array([grains_refl[f].eps_sample_matrix(struc.lattice_parameters) for f in fit_refl])
strains_both = jnp.array([grains_both[f].eps_sample_matrix(struc.lattice_parameters) for f in fit_both])

strains_norm_dir  = jnp.sqrt((strains_dir**2).sum(axis=(1,2)))
strains_norm_refl = jnp.sqrt((strains_refl**2).sum(axis=(1,2)))
strains_norm_both = jnp.sqrt((strains_both**2).sum(axis=(1,2)))

loss_diff_dir  = all_losses_dir[fit_dir, -1]   - gt_losses_dir
loss_diff_refl = all_losses_refl[fit_refl, -1] - gt_losses_refl
loss_diff_both = all_losses_both[fit_both, -1] - gt_losses_both

print("refined:", float(all_losses_dir[fit_dir, -1].mean()),
      " ground truth:", float(gt_losses_dir.mean()))

In [ ]:
fig, axs = plt.subplots(3,1,sharex=True,sharey=True,constrained_layout=True)
axs[0].hist(pos_diff_dir, bins=20)
axs[1].hist(pos_diff_refl, bins=20)
axs[2].hist(pos_diff_both, bins=20)
axs[0].set_title(r'Just direct beam')
axs[1].set_title(r'Just reflected beam')
axs[2].set_title(r'Direct and reflected beams')
fig.supxlabel(r'Position error to ground truth (μm)')
fig.supylabel('Counts')
plt.show()

In [ ]:
fig, axs = plt.subplots(3,1,sharex=True,sharey=True,constrained_layout=True)
axs[0].hist(strains_norm_dir*1e3, bins=20)
axs[1].hist(strains_norm_refl*1e3, bins=20)
axs[2].hist(strains_norm_both*1e3, bins=20)
axs[0].set_title(r'Just direct beam')
axs[1].set_title(r'Just reflected beam')
axs[2].set_title(r'Direct and reflected beams')
fig.supxlabel(r'Strain norm (should be 0) (x1e-3)')
fig.supylabel('Counts')
plt.show()

In [ ]:
fig, axs = plt.subplots(3,1,sharex=True,sharey=True,constrained_layout=True)
axs[0].hist(misorien_dir, bins=20)
axs[1].hist(misorien_refl, bins=20)
axs[2].hist(misorien_both, bins=20)
axs[0].set_title(r'Just direct beam')
axs[1].set_title(r'Just reflected beam')
axs[2].set_title(r'Direct and reflected beams')
fig.supxlabel('Misorientation to ground truth (deg)')
fig.supylabel('Counts')
plt.show()

In [ ]:
fig, axs = plt.subplots(3,1,sharex=True,sharey=True,constrained_layout=True)
axs[0].hist(all_losses_dir[:, -1], bins=20, alpha=0.5)
axs[0].hist(gt_losses_dir, bins=20, alpha=0.5)
axs[1].hist(all_losses_refl[:, -1], bins=20, alpha=0.5)
axs[1].hist(gt_losses_refl, bins=20,  alpha=0.5)
axs[2].hist(all_losses_both[:, -1], bins=20,alpha=0.5)
axs[2].hist(gt_losses_both, bins=20, alpha=0.5)
axs[0].set_title(r'Just direct beam')
axs[1].set_title(r'Just reflected beam')
axs[2].set_title(r'Direct and reflected beams')
fig.supxlabel('Final loss function')
fig.supylabel('Counts')
plt.show()

In [ ]:
fig, axs = plt.subplots(3,1,sharex=True,sharey=True,constrained_layout=True)
axs[0].hist(loss_diff_dir, bins=20)
axs[1].hist(loss_diff_refl, bins=20)
axs[2].hist(loss_diff_both, bins=20)
axs[0].set_title(r'Just direct beam')
axs[1].set_title(r'Just reflected beam')
axs[2].set_title(r'Direct and reflected beams')
fig.supxlabel('Difference in loss function to ground truth')
fig.supylabel('Counts')
plt.show()

In [ ]:
def print_grain_metrics(
    pos_diff_dir, pos_diff_refl, pos_diff_both,
    misorien_dir, misorien_refl, misorien_both,
    strains_norm_dir, strains_norm_refl, strains_norm_both,
    loss_diff_dir, loss_diff_refl, loss_diff_both
):
    # Defining metrics with their specific scaling factors
    # (Label, datdir, datrefl, databoth, scale_factor, unit_label)
    metrics = [
        ("Loss Diff", loss_diff_dir, loss_diff_refl, loss_diff_both, 1e6, "(x1e-6)"),
        ("Positional Diff", pos_diff_dir, pos_diff_refl, pos_diff_both, 1.0, ""),
        ("Misorientation", misorien_dir, misorien_refl, misorien_both, 1e3, "(x1e-3)"),
        ("Strain Norm", strains_norm_dir, strains_norm_refl, strains_norm_both, 1e4, "(x1e-4)"),
    ]

    header = f"{'Metric':<25} | {'dir Mean':<12} | {'refl Mean':<12} | {'Both Mean':<12} | {'Comb. Frac':<12}"
    print(header)
    print("-" * len(header))

    for label, d1, d2, db, scale, unit in metrics:
        # Apply scaling and calculate means
        m1 = jnp.mean(d1).item() * scale
        m2 = jnp.mean(d2).item() * scale
        mb = jnp.mean(db).item() * scale
        
        m_avg_indiv = (m1 + m2) / 2
        
        # The fraction remains the same regardless of scaling
        fraction = mb / m_avg_indiv if m_avg_indiv != 0 else 0
        
        full_label = f"{label} {unit}".strip()
        print(f"{full_label:<25} | {m1:>12.6f} | {m2:>12.6f} | {mb:>12.6f} | {fraction:>12.4f}")

print_grain_metrics(
    pos_diff_dir, pos_diff_refl, pos_diff_both,
    misorien_dir, misorien_refl, misorien_both,
    strains_norm_dir, strains_norm_refl, strains_norm_both,
    loss_diff_dir, loss_diff_refl, loss_diff_both
)

In [ ]:
# total peaks (dir and refl)
mask_dir.sum(), mask_dir.sum()*2, mask_dir.sum()/n_grains, mask_dir.sum()*2/n_grains

## Plots for paper

In [ ]:
fig, ax = plt.subplots(figsize=(3.54, 3.54), layout='constrained')
ax.scatter(centroid_with_error[:mask_dir.sum()][:, 1], centroid_with_error[:mask_dir.sum()][:, 0], label=r'Direct peaks', s=.5)
ax.scatter(centroid_with_error[mask_dir.sum():][:, 1], centroid_with_error[mask_dir.sum():][:, 0], label=r'Reflected peaks', s=.5)
ax.set_aspect(1)
ax.set(title='Grazing incidence')
ax.set_xticks([])
ax.set_yticks([])
ax.legend(loc='upper right')
plt.show()
plt.savefig('graz_peaks.png', dpi=600)

In [ ]:
end = time.time()
print(f'Took {end-start:.0f} seconds')